In [ ]:
from datetime import datetime
from dateutil.relativedelta import relativedelta

import pandas as pd
pd.set_option("display.max_columns", 30)

import requests

In [ ]:
current_datetime = datetime.now() - relativedelta(months=2)
forrmatted_datetime = current_datetime.strftime("%Y-%m-%d")

url =( 
    f"https://data.cityofchicago.org/resource/ajtu-isnz.json?"
    f"$where=trip_start_timestamp >= '{forrmatted_datetime}T00:00:00' "
    f"AND trip_start_timestamp <= '{forrmatted_datetime}T23:59:59' "
    f"&$limit=50000"
)
response = requests.get(url)
data = response.json()

len(data)

In [ ]:
taxi_trips = pd.DataFrame(data)

taxi_trips.head()

In [ ]:
taxi_trips.info()

In [ ]:
taxi_trips.describe()

In [ ]:
taxi_trips[taxi_trips["fare"].isna()].sample(5)

#### Transformation: deal with NaN values

In [ ]:
taxi_trips.drop(["pickup_census_tract", "dropoff_census_tract"], axis=1, inplace=True)

In [ ]:
taxi_trips.drop(["pickup_centroid_location", "dropoff_centroid_location"], axis=1, inplace=True)

In [ ]:
taxi_trips.dropna(inplace=True)

In [ ]:
taxi_trips.info()

#### Transformation: renaming

In [ ]:
taxi_trips.rename(columns={"pickup_community_area": "pickup_community_area_id",
                           "dropoff_community_area": "dropoff_community_area_id"}, inplace=True)

In [ ]:
taxi_trips.columns

#### Transformation: create helper column for weather and date dimension

In [ ]:
taxi_trips["trip_start_timestamp"] = pd.to_datetime(taxi_trips["trip_start_timestamp"])

In [ ]:
taxi_trips["datetime_for_weather"] = taxi_trips["trip_start_timestamp"].dt.floor("h")

In [ ]:
taxi_trips["datetime_for_weather"].unique()

In [ ]:
taxi_trips["trip_start_date"] = pd.to_datetime(taxi_trips["trip_start_timestamp"]).dt.date

In [ ]:
taxi_trips["trip_start_date"].sample(5)

#### Check: join taxi and weather data

In [ ]:
current_datetime = datetime.now() - relativedelta(months=2)
formatted_datetime = current_datetime.strftime("%Y-%m-%d")

url = "https://archive-api.open-meteo.com/v1/era5"

params = {
    "latitude": 41.85,
    "longitude": -87.65,
    "start_date": formatted_datetime,
    "end_date": formatted_datetime,
    "hourly": "temperature_2m,wind_speed_10m,rain,precipitation"
}

response = requests.get(url, params=params)
data = response.json()

weather_data = {
    "datetime": data["hourly"]["time"],
    "teperature": data["hourly"]["temperature_2m"],
    "wind_speed": data["hourly"]["wind_speed_10m"],
    "rain": data["hourly"]["rain"],
    "percipitation": data["hourly"]["precipitation"]

}

weather_df = pd.DataFrame(weather_data)
weather_df["datetime"] = pd.to_datetime(weather_df["datetime"])


weather_df.head()

In [ ]:
taxi_trips_with_weather = taxi_trips.merge(weather_df, left_on="datetime_for_weather", right_on="datetime")

In [ ]:
taxi_trips_with_weather.head()

#### Transformation: data type conversions

In [ ]:
taxi_trips.info()

In [ ]:
taxi_trips_originial = taxi_trips.copy()

In [ ]:
data_type = {
    "trip_end_timestamp": "datetime64[ns]",
    "trip_seconds": "int32",
    "trip_miles": "float",
    "pickup_community_area_id": "int8",
    "dropoff_community_area_id": "int8",
    "fare": "float",
    "tips": "float",
    "tolls": "float",
    "extras": "float",
    "trip_total": "float",
    "pickup_centroid_latitude": "float",
    "pickup_centroid_longitude": "float",
    "dropoff_centroid_latitude": "float",
    "dropoff_centroid_longitude": "float",
}

taxi_trips = taxi_trips.astype(data_type)

In [ ]:
taxi_trips.info()

In [ ]:
taxi_trips.describe()

##### Memory usage

In [ ]:
orig_mem = taxi_trips_originial.memory_usage(deep=True).sum()
opt_mem = taxi_trips.memory_usage(deep=True).sum()

reduction = (orig_mem - opt_mem) / orig_mem * 100

print(f"Original: {orig_mem:,} bytes")
print(f"Optimzed: {opt_mem:,} bytes")
print(f"Memory reduced by: {reduction:.2f}%")

#### Sanity checks

In [ ]:
taxi_trips[taxi_trips["trip_end_timestamp"] == taxi_trips["trip_end_timestamp"].max()]

In [ ]:
taxi_trips[taxi_trips["trip_seconds"] == taxi_trips["trip_seconds"].max()]

In [ ]:
taxi_trips[taxi_trips["fare"] == taxi_trips["fare"].max()]

#### Creating diemsnion tables for Comapny and Payment Type

##### Payment Type Dimension

In [ ]:
dim_payment_type = taxi_trips["payment_type"].drop_duplicates().reset_index(drop=True)

dim_payment_type =pd.DataFrame(
    {
        "payment_type_id": range(1, len(dim_payment_type) + 1),
        "payment_type": dim_payment_type
    }
)

dim_payment_type

##### Company dimensio

In [ ]:
dim_company = taxi_trips["company"].drop_duplicates().reset_index(drop=True)

dim_company =pd.DataFrame(
    {
        "company_id": range(1, len(dim_company) + 1),
        "company": dim_company
    }
)

dim_company

##### Merging back to the Fact table

In [ ]:
fact_taxi_trips = taxi_trips.merge(dim_payment_type, on="payment_type")
fact_taxi_trips = fact_taxi_trips.merge(dim_company, on="company")

In [ ]:
fact_taxi_trips.sample(5)

In [ ]:
fact_taxi_trips.drop(["payment_type", "company"], axis=1, inplace=True)

In [ ]:
fact_taxi_trips.sample(5)

In [ ]:
dim_payment_type.to_csv("dim_payment_type.csv", index=False)
dim_company.to_csv("dim_company.csv", index=False)


##### Create update logic for Dimension tables

In [ ]:
dim_payment_type

In [ ]:
dim_payment_type = taxi_trips["payment_type"].drop_duplicates().reset_index(drop=True)

dim_payment_type =pd.DataFrame(
    {
        "payment_type_id": range(1, len(dim_payment_type) + 1),
        "payment_type": dim_payment_type
    }
)
dummy_payment_type_date = [
    {"payment_type": "Credit card"},
    {"payment_type": "X"},
    {"payment_type": "Y"},
    {"payment_type": "X"},
]
dummy_payment_type_date_df = pd.DataFrame(dummy_payment_type_date)


todays_payment_types = pd.DataFrame(dummy_payment_type_date_df["payment_type"].unique(), columns=["payment_type"])
new_payment_types = todays_payment_types[~todays_payment_types["payment_type"].isin(dim_payment_type["payment_type"])]

if not new_payment_types.empty:
    max_id = dim_payment_type["payment_type_id"].max()
    new_payment_types["payment_type_id"] = range(max_id + 1, max_id + 1 + len(new_payment_types))
    dim_payment_type = pd.concat([dim_payment_type, new_payment_types], ignore_index=True)

dim_payment_type

In [ ]:
dummy_payment_type_date = [
    {"payment_type": "Credit card"},
    {"payment_type": "X"},
    {"payment_type": "Z"},
    {"payment_type": "Z"},
]
dummy_payment_type_date_df = pd.DataFrame(dummy_payment_type_date)

todays_payment_types = pd.DataFrame(dummy_payment_type_date_df["payment_type"].unique(), columns=["payment_type"])
new_payment_types = todays_payment_types[~todays_payment_types["payment_type"].isin(dim_payment_type["payment_type"])]

if not new_payment_types.empty:
    max_id = dim_payment_type["payment_type_id"].max()
    new_payment_types["payment_type_id"] = range(max_id + 1, max_id + 1 + len(new_payment_types))
    dim_payment_type = pd.concat([dim_payment_type, new_payment_types], ignore_index=True)

dim_payment_type

In [ ]:
dim_company = taxi_trips["company"].drop_duplicates().reset_index(drop=True)

dim_company =pd.DataFrame(
    {
        "company_id": range(1, len(dim_company) + 1),
        "company": dim_company
    }
)
dummy_company_date = [
    {"company": "Metro Jet Taxi A."},
    {"company": "X"},
    {"company": "Y"},
    {"company": "X"},
]
todays_companies_data_df = pd.DataFrame(dummy_company_date)


todays_companies = pd.DataFrame(todays_companies_data_df["company"].unique(), columns=["company"])
new_companies = todays_companies[~todays_companies["company"].isin(dim_company["company"])]

if not new_companies.empty:
    max_id = dim_company["company_id"].max()
    new_companies["company_id"] = range(max_id + 1, max_id + 1 + len(new_companies))
    dim_company = pd.concat([dim_company, new_companies], ignore_index=True)

dim_company.tail()